# 🤖 What is an AI Agent?

An **AI Agent** is essentially a function you write to perform specific tasks that an LLM cannot do on its own.

## Why Do We Need AI Agents?

LLMs accessed via API have limitations:

* **Calculations**: When you ask for a calculation, the LLM will only *predict* the answer based on patterns, not perform actual computation. This can lead to inaccurate results.
* **Real-time Data**: LLMs cannot access live information like:
  * Current weather in a city
  * Live cryptocurrency prices
  * Stock market data
  * Database queries
  * File system operations

## How AI Agents Work

You create functions in your Python code that:
1. Get executed when the user asks specific types of questions
2. Perform the actual computation or data retrieval
3. Return accurate, real-time results to the LLM
4. Allow the LLM to incorporate this data into its response

## Terminology

These functions are known by several names:
* **AI Agents**
* **Tools**
* **Functions**
* **Function Calling**

## Example Use Cases

| User Request | AI Agent Function |
|--------------|-------------------|
| "Calculate 2847 × 3921" | `calculator(expression)` |
| "What's the weather in New York?" | `get_weather(city)` |
| "What's the current Bitcoin price?" | `get_crypto_price(symbol)` |
| "Search my emails for invoices" | `search_emails(query)` |
| "Query our sales database" | `execute_sql_query(sql)` |

![image_1776939199413.png](./image_1776939199413.png "AI Agent Architecture Diagram")

## Key Takeaway

AI Agents bridge the gap between what LLMs can predict (text generation) and what they cannot do (computation, real-time data access, external system integration).






In [0]:
%pip install groq

import sys
import json
from groq import Groq
import os

# Add parent folder (one level above GenAI-AgenticAI) to Python path to import config
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('')))
sys.path.insert(0, parent_dir)

# Import API key from config file (stored outside git repo)
from groq_config import GROQ_API_KEY

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# ═══════════════════════════════════════════════════════════════
# Step 1: Define the AI Agent (Tool/Function)
# ═══════════════════════════════════════════════════════════════

def calculate(expression: str) -> float:
    """AI Agent that performs mathematical calculations."""
    try:
        # Safely evaluate the mathematical expression
        result = eval(expression, {"__builtins__": {}}, {})
        return float(result)
    except Exception as e:
        return f"Error: {str(e)}"

# ═══════════════════════════════════════════════════════════════
# Step 2: Define Tool Schema for the LLM
# ═══════════════════════════════════════════════════════════════

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Performs mathematical calculations. Use this when the user asks for any calculation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to evaluate (e.g., '2847 * 3921')"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# ═══════════════════════════════════════════════════════════════
# Step 3: Ask the LLM a Question That Requires Tool Use
# ═══════════════════════════════════════════════════════════════

user_question = "What is 2847 multiplied by 3921?"

print("="*60)
print(f"🧑 USER: {user_question}")
print("="*60)

# Call the LLM with the tools available
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": user_question}],
    tools=tools,
    tool_choice="auto"  # Let the model decide if/when to use tools
)

message = response.choices[0].message
print(message)
# ═══════════════════════════════════════════════════════════════
# Step 4: Check if the LLM Wants to Use a Tool
# ═══════════════════════════════════════════════════════════════

if message.tool_calls:
    print("\n🤖 LLM DECISION: I need to use a tool to calculate this!\n")
    
    # Extract the tool call details
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    
    print(f"📞 TOOL CALL:")
    print(f"   Function: {function_name}")
    print(f"   Arguments: {function_args}")
    
    # ═══════════════════════════════════════════════════════════════
    # Step 5: Execute the Tool (AI Agent)
    # ═══════════════════════════════════════════════════════════════
    
    if function_name == "calculate":
        result = calculate(function_args["expression"])
        print(f"\n⚙️  TOOL EXECUTION: calculate('{function_args['expression']}')")
        print(f"   Result: {result}")
    
    # ═══════════════════════════════════════════════════════════════
    # Step 6: Send Tool Result Back to LLM for Final Response
    # ═══════════════════════════════════════════════════════════════
    
    # Prepare messages with tool result
    messages = [
        {"role": "user", "content": user_question},
        message,  # The assistant's decision to use the tool
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result)
        }
    ]
    
    # Get final response from LLM
    final_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )
    
    print(f"\n🤖 LLM FINAL RESPONSE: {final_response.choices[0].message.content}")
    
else:
    # LLM responded directly without using tools
    print(f"\n🤖 LLM RESPONSE (no tool used): {message.content}")

print("\n" + "="*60)
print("✅ AI Agent Demo Complete!")
print("="*60)